# ISOM5240 Fine-tuning Notebook — Pipeline 1: Shelf-life Classification

**Workflow:**
- Phase 1 → Quick 1-epoch fine-tune of 3 models on small subset → select best
- Phase 2 → Full fine-tune of selected model on complete dataset
- Phase 3 → Evaluate + export Excel + push to HuggingFace Hub

**Datasets:**
- Grocery Store Dataset (GitHub) → short_shelf + medium_shelf
- Kaggle Household Products → non_perishable

**Task:** 3-class image classification (short_shelf / medium_shelf / non_perishable)

## Step 1: Install dependencies

In [ ]:
!pip install transformers datasets evaluate accelerate pillow scikit-learn -q
!pip install huggingface_hub -q

## Step 2: GPU check

In [ ]:
import torch
import os
import time
import numpy as np

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Runtime → Change runtime type → T4 GPU → Save, then Run All.")

print(f"Using GPU: {torch.cuda.get_device_name(0)}")

## Step 3: Login to HuggingFace + setup Kaggle

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
print("HuggingFace + Kaggle ready")

## Step 4: Download both datasets

In [ ]:
# Grocery Store Dataset (fruits, vegetables, dairy)
!git clone https://github.com/marcusklasson/GroceryStoreDataset.git

# Household Products (non-perishable items)
!kaggle datasets download -d taru149/householdproducts
!unzip -q householdproducts.zip -d household/

print("Both datasets downloaded!")
print("\nGrocery Store test folders:")
print(os.listdir("GroceryStoreDataset/dataset/test"))
print("\nHousehold folders:")
!find household/ -type d | head -15

## Step 5: Build 3-class dataset

- short_shelf (0): fruits, vegetables
- medium_shelf (1): dairy, juice, milk
- non_perishable (2): household products

In [ ]:
from datasets import Dataset, DatasetDict
from PIL import Image
import glob

# Keywords for Grocery Store Dataset folder → shelf-life mapping
SHORT_KEYWORDS = [
    "apple", "avocado", "banana", "kiwi", "lemon", "lime", "mango",
    "melon", "nectarine", "orange", "papaya", "passion", "peach",
    "pear", "pineapple", "plum", "pomegranate", "grapefruit",
    "satsuma", "watermelon", "asparagus", "aubergine", "cabbage",
    "carrot", "cucumber", "garlic", "ginger", "leek", "mushroom",
    "onion", "pepper", "potato", "red-beet", "tomato", "zucchini"
]

MEDIUM_KEYWORDS = [
    "juice", "milk", "oat", "sour-cream", "sour-milk",
    "soy", "yoghurt", "cream"
]

LABEL_NAMES = ["short_shelf", "medium_shelf", "non_perishable"]

def collect_grocery_images(split_dir):
    images = []
    labels = []
    for class_dir in sorted(os.listdir(split_dir)):
        class_path = os.path.join(split_dir, class_dir)
        if not os.path.isdir(class_path):
            continue
        name = class_dir.lower()
        if any(kw in name for kw in SHORT_KEYWORDS):
            label = 0
        elif any(kw in name for kw in MEDIUM_KEYWORDS):
            label = 1
        else:
            continue
        for f in os.listdir(class_path):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                images.append(os.path.join(class_path, f))
                labels.append(label)
    return images, labels

# Collect grocery train + test
grocery_train_imgs, grocery_train_labels = collect_grocery_images("GroceryStoreDataset/dataset/train")
grocery_test_imgs, grocery_test_labels = collect_grocery_images("GroceryStoreDataset/dataset/test")

# Collect household (all = non_perishable = label 2)
household_imgs = (glob.glob("household/**/*.jpg", recursive=True) +
                  glob.glob("household/**/*.png", recursive=True) +
                  glob.glob("household/**/*.jpeg", recursive=True))

print(f"Grocery train: {len(grocery_train_imgs)} | Grocery test: {len(grocery_test_imgs)}")
print(f"Household: {len(household_imgs)}")

In [ ]:
# Split household into train/test (80/20)
from sklearn.model_selection import train_test_split

if len(household_imgs) > 0:
    hh_train, hh_test = train_test_split(household_imgs, test_size=0.2, random_state=42)
else:
    hh_train, hh_test = [], []

# Combine into final datasets
all_train_paths = grocery_train_imgs + hh_train
all_train_labels = grocery_train_labels + [2] * len(hh_train)

all_test_paths = grocery_test_imgs + hh_test
all_test_labels = grocery_test_labels + [2] * len(hh_test)

print(f"Combined train: {len(all_train_paths)}")
print(f"Combined test: {len(all_test_paths)}")

# Count per class
all_train_labels_np = np.array(all_train_labels)
all_test_labels_np = np.array(all_test_labels)
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name}: train={int((all_train_labels_np==i).sum())}, test={int((all_test_labels_np==i).sum())}")

In [ ]:
# Build HuggingFace Dataset objects
def load_image(path):
    return Image.open(path).convert("RGB")

train_dataset = Dataset.from_dict({
    "image": [load_image(p) for p in all_train_paths],
    "label": all_train_labels,
})

test_dataset = Dataset.from_dict({
    "image": [load_image(p) for p in all_test_paths],
    "label": all_test_labels,
})

# Split validation from train
split = train_dataset.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": test_dataset,
})

print(f"Train:      {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test:       {len(dataset['test'])}")

---
# PHASE 1: Model Selection

Quick 1-epoch fine-tune of each model on small subset to fairly compare.

## Step 6: Define candidates

In [ ]:
CANDIDATE_MODELS = {
    "ViT-base": "google/vit-base-patch16-224",
    "ResNet-50": "microsoft/resnet-50",
    "Swin-tiny": "microsoft/swin-tiny-patch4-window7-224",
}

print("Candidates:")
for name, path in CANDIDATE_MODELS.items():
    print(f"  {name}: {path}")

## Step 7: Quick 1-epoch comparison

In [ ]:
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    pipeline as hf_pipeline,
)
import evaluate
import pandas as pd

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

small_train = dataset["train"].select(range(min(500, len(dataset["train"]))))
small_test = dataset["test"].select(range(min(200, len(dataset["test"]))))

quick_results = []

for model_key, model_path in CANDIDATE_MODELS.items():
    print(f"\n{'='*50}")
    print(f"Quick training: {model_key}")
    print(f"{'='*50}")

    proc = AutoImageProcessor.from_pretrained(model_path)

    def make_preprocess(processor):
        def preprocess(batch):
            images = [img.convert("RGB") for img in batch["image"]]
            inputs = processor(images=images, return_tensors="pt")
            inputs["label"] = batch["label"]
            return inputs
        return preprocess

    transform_fn = make_preprocess(proc)
    small_train.set_transform(transform_fn)
    small_test.set_transform(transform_fn)

    mdl = AutoModelForImageClassification.from_pretrained(
        model_path,
        num_labels=3,
        id2label={0: "short_shelf", 1: "medium_shelf", 2: "non_perishable"},
        label2id={"short_shelf": 0, "medium_shelf": 1, "non_perishable": 2},
        ignore_mismatched_sizes=True,
    )

    total_params = sum(p.numel() for p in mdl.parameters())

    args = TrainingArguments(
        output_dir=f"./quick-p1-{model_key}",
        num_train_epochs=1,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        fp16=True,
        remove_unused_columns=False,
        logging_steps=999,
        report_to="none",
    )

    trainer = Trainer(
        model=mdl, args=args,
        train_dataset=small_train,
        eval_dataset=small_test,
        compute_metrics=compute_metrics,
    )

    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    results = trainer.evaluate()

    # Measure inference speed
    test_pipe = hf_pipeline("image-classification", model=mdl, image_processor=proc, device=0)
    sample_paths = all_test_paths[:50]
    inf_times = []
    for path in sample_paths:
        img = Image.open(path).convert("RGB")
        t0 = time.time()
        test_pipe(img)
        inf_times.append(time.time() - t0)
    avg_inf_ms = np.mean(inf_times) * 1000

    quick_results.append({
        "Model": model_key,
        "Parameters (M)": f"{total_params/1e6:.1f}M",
        "1-Epoch Accuracy": round(results["eval_accuracy"], 4),
        "Train Time (s)": round(train_time, 1),
        "Avg Inference (ms)": round(avg_inf_ms, 1),
    })

    print(f"  Accuracy: {results['eval_accuracy']:.4f} | Time: {train_time:.0f}s | Speed: {avg_inf_ms:.1f}ms")

df_selection = pd.DataFrame(quick_results)
print("\n" + "="*60)
print("PHASE 1 RESULTS: Model Selection (3-class, 1-epoch, small subset)")
print("="*60)
print(df_selection.to_string(index=False))

## Step 8: Select best model

In [ ]:
best = max(quick_results, key=lambda x: x["1-Epoch Accuracy"])
SELECTED_MODEL_KEY = best["Model"]
SELECTED_MODEL_PATH = CANDIDATE_MODELS[SELECTED_MODEL_KEY]

print(f"Selected: {SELECTED_MODEL_KEY} (Accuracy: {best['1-Epoch Accuracy']})")

---
# PHASE 2: Full Fine-tuning

## Step 9: Prepare data

In [ ]:
processor = AutoImageProcessor.from_pretrained(SELECTED_MODEL_PATH)

def preprocess(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    inputs = processor(images=images, return_tensors="pt")
    inputs["label"] = batch["label"]
    return inputs

dataset["train"].set_transform(preprocess)
dataset["validation"].set_transform(preprocess)
dataset["test"].set_transform(preprocess)

print(f"Transform set for {SELECTED_MODEL_KEY}")

## Step 10: Load model (3-class head)

In [ ]:
model = AutoModelForImageClassification.from_pretrained(
    SELECTED_MODEL_PATH,
    num_labels=len(LABEL_NAMES),
    id2label={i: l for i, l in enumerate(LABEL_NAMES)},
    label2id={l: i for i, l in enumerate(LABEL_NAMES)},
    ignore_mismatched_sizes=True,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {SELECTED_MODEL_KEY} | Params: {total_params:,}")

## Step 11: Train

In [ ]:
training_args = TrainingArguments(
    output_dir=f"./shelf-life-{SELECTED_MODEL_KEY.lower()}",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    remove_unused_columns=False,
    fp16=True,
    dataloader_num_workers=2,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
)

train_start = time.time()
trainer.train()
train_time = time.time() - train_start
print(f"\nTraining complete in {train_time/60:.1f} minutes")

## Step 12: Evaluate — Accuracy

In [ ]:
ft_results = trainer.evaluate(dataset["test"])
print(f"Test Accuracy: {ft_results['eval_accuracy']:.4f}")
print(f"Test Loss:     {ft_results['eval_loss']:.4f}")

## Step 13: Evaluate — Precision, Recall, F1, Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image as PILImage

pipe_ft = hf_pipeline("image-classification", model=model, image_processor=processor, device=0)

y_true = []
y_pred = []

for i, path in enumerate(all_test_paths):
    img = PILImage.open(path).convert("RGB")
    true_label = LABEL_NAMES[all_test_labels[i]]

    pred = pipe_ft(img, top_k=1)
    pred_label = pred[0]["label"]

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Classification Report:")
print(classification_report(y_true, y_pred, digits=4))

print("Confusion Matrix:")
cm = confusion_matrix(y_true, y_pred, labels=LABEL_NAMES)
print(cm)

## Step 14: Inference speed

In [ ]:
inf_times = []
for path in all_test_paths[:100]:
    img = PILImage.open(path).convert("RGB")
    t0 = time.time()
    pipe_ft(img)
    inf_times.append(time.time() - t0)

ft_avg_ms = np.mean(inf_times) * 1000
print(f"Avg inference: {ft_avg_ms:.1f}ms per image")

## Step 15: Before vs After comparison

In [ ]:
selected_phase1 = next(r for r in quick_results if r["Model"] == SELECTED_MODEL_KEY)

comparison = pd.DataFrame([
    {
        "Stage": "Quick (1-epoch, small subset)",
        "Model": SELECTED_MODEL_KEY,
        "Accuracy": selected_phase1["1-Epoch Accuracy"],
        "Avg Inference (ms)": selected_phase1["Avg Inference (ms)"],
    },
    {
        "Stage": "Full fine-tuned",
        "Model": SELECTED_MODEL_KEY,
        "Accuracy": round(ft_results["eval_accuracy"], 4),
        "Avg Inference (ms)": round(ft_avg_ms, 1),
    },
])

print("="*60)
print("Before vs After Fine-tuning")
print("="*60)
print(comparison.to_string(index=False))

## Step 16: Export Excel

In [ ]:
with pd.ExcelWriter("P1_Experimental_results.xlsx") as writer:
    df_selection.to_excel(writer, sheet_name="P1 Model Selection", index=False)
    comparison.to_excel(writer, sheet_name="P1 Fine-tune Result", index=False)

print("Saved!")

from google.colab import files
files.download("P1_Experimental_results.xlsx")

## Step 17: Push to HuggingFace Hub

In [ ]:
HUB_MODEL_ID = "Alisa-Sun/shelf-life-classification"  # TODO: Change if needed

model.push_to_hub(HUB_MODEL_ID, commit_message=f"Fine-tuned {SELECTED_MODEL_KEY} | acc={ft_results['eval_accuracy']:.4f}")
processor.push_to_hub(HUB_MODEL_ID)

print(f"Model pushed to: https://huggingface.co/{HUB_MODEL_ID}")

## Step 18: Final inference test

In [ ]:
pipe_final = hf_pipeline("image-classification", model=HUB_MODEL_ID)

print("Testing 5 images:")
for i in range(min(5, len(all_test_paths))):
    img = PILImage.open(all_test_paths[i]).convert("RGB")
    true_label = LABEL_NAMES[all_test_labels[i]]
    pred = pipe_final(img)
    status = "✅" if pred[0]["label"] == true_label else "❌"
    print(f"  {status} True: {true_label} | Pred: {pred[0]['label']} ({pred[0]['score']:.3f})")

---
## Summary

| Step | Content |
|------|---------|
| 1-3 | Setup |
| 4-5 | Download Grocery Store + Household Products → build 3-class dataset |
| 6-7 | Phase 1: quick 1-epoch comparison of ViT/ResNet/Swin |
| 8 | Select best model |
| 9-11 | Phase 2: full fine-tune (3 epochs) |
| 12-14 | Evaluate: accuracy, precision, recall, confusion matrix, speed |
| 15 | Before vs after comparison |
| 16 | Export Excel |
| 17-18 | Push to Hub + final test |